# AHS-KT × BD2006 可运行 Notebook

这个 Notebook 面向 `/root/autodl-tmp/ahs-kt/data/BD06`，目标是把 **BD2006** 在当前仓库的 `AHSKTModel` 上完整跑通，并输出：

- `auc`
- `acc`
- `rmse`

## 这本 Notebook 采用的评估协议

BD2006 目录里有三个关键文件：

- `bridge_to_algebra_2006_2007_train.txt`：官方带标签训练日志
- `bridge_to_algebra_2006_2007_test.txt`：官方公开 test 行号，但**没有标签**
- `bridge_to_algebra_2006_2007_master.txt`：和公开 test 对齐、但带标签的评分文件

这里有两个容易踩坑的地方：

1. `test.txt` 本身没有标签，所以不能直接算 `auc/acc/rmse`。
2. `master.txt` 的测试点并不是简单地“全部晚于 train”，而是和 train 时间轴交错出现。

因此这里采用的方案是：

- 保持 **question-level** 时间步，不把一条原始行按多 KC 展开，避免同题标签泄漏；
- `KC(SubSkills)` 为空时映射到特殊概念 `__NO_KC__`，避免丢掉大量样本；
- validation 从 `train.txt` 每个学生时间轴尾部拿 5 个点，并且这些点**不作为可观察历史**；
- test 使用 `master.txt` 的标签，但每个测试点只允许看到**更早的官方 train 交互**，不允许看到其他 held-out test 标签。

如果 `outputs/ahskt_bd2006_notebook/` 下结果已经存在，这本 Notebook 会直接复用；否则会从头构建 bundle、训练并导出结果。

In [1]:
from pathlib import Path
import json
import os
import sys

PROJECT_ROOT = Path('/root/autodl-tmp/ahs-kt')
SRC_ROOT = PROJECT_ROOT / 'src'
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

os.chdir(PROJECT_ROOT)
print('PROJECT_ROOT =', PROJECT_ROOT)
print('BD06_DIR =', PROJECT_ROOT / 'data' / 'BD06')

PROJECT_ROOT = /root/autodl-tmp/ahs-kt
BD06_DIR = /root/autodl-tmp/ahs-kt/data/BD06


In [2]:
from scripts.run_bd2006_ahskt import BD2006RunConfig, run_bd2006_experiment

run_config = BD2006RunConfig(
    task_name='ahskt_bd2006_notebook',
    epochs=3,
    batch_size=64,
    seed=2026,
    output_root='outputs/ahskt_bd2006_notebook',
    data_output_root='data/ahskt_bd2006_notebook',
    config_output_path='configs/ahskt_bd2006_notebook.json',
)

metrics_path = PROJECT_ROOT / run_config.output_root / f'{run_config.task_name}_metrics.json'
manifest_path = PROJECT_ROOT / run_config.output_root / f'{run_config.task_name}_manifest.json'

if metrics_path.exists() and manifest_path.exists():
    print('检测到已有结果，直接复用。')
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    metrics_summary = json.loads(metrics_path.read_text(encoding='utf-8'))
    result = {
        'config_path': manifest['config_path'],
        'metrics_path': str(metrics_path),
        'manifest_path': str(manifest_path),
        'saved_paths': manifest['saved_paths'],
        'prepared_metadata': manifest['prepared_metadata'],
        'metrics_summary': metrics_summary,
    }
else:
    print('未发现已有结果，开始从头运行 BD2006 实验。')
    result = run_bd2006_experiment(run_config)
    manifest = json.loads(Path(result['manifest_path']).read_text(encoding='utf-8'))
    metrics_summary = result['metrics_summary']

result['metrics_summary']['test_metrics']

检测到已有结果，直接复用。


{'loss': 0.3652456998825073,
 'auc': 0.8081964418440444,
 'acc': 0.8511276235171424,
 'rmse': 0.3338070511817932}

In [3]:
import pandas as pd
from IPython.display import display

split_summary = pd.DataFrame([result['prepared_metadata']['split_summary']]).T.reset_index()
split_summary.columns = ['item', 'value']
metrics_df = pd.DataFrame([metrics_summary['test_metrics']])
history_df = pd.DataFrame([
    {
        'epoch': row['epoch'],
        'train_auc': row['train']['auc'],
        'train_acc': row['train']['acc'],
        'train_rmse': row['train']['rmse'],
        'valid_auc': row['valid']['auc'],
        'valid_acc': row['valid']['acc'],
        'valid_rmse': row['valid']['rmse'],
    }
    for row in metrics_summary['history']
])

print('best_epoch =', metrics_summary['best_epoch'])
print('best_valid_auc =', metrics_summary['best_valid_auc'])
print('checkpoint_path =', metrics_summary['checkpoint_path'])
print('config_path =', result['config_path'])
print('metrics_path =', result['metrics_path'])
print('manifest_path =', result['manifest_path'])

print()
print('Test metrics:')
display(metrics_df)

print()
print('Split summary:')
display(split_summary)

print()
print('Training history:')
display(history_df)


best_epoch = 3
best_valid_auc = 0.817870979047064
checkpoint_path = /root/autodl-tmp/ahs-kt/outputs/ahskt_bd2006_notebook/checkpoints/ckpt-3
config_path = /root/autodl-tmp/ahs-kt/configs/ahskt_bd2006_notebook.json
metrics_path = /root/autodl-tmp/ahs-kt/outputs/ahskt_bd2006_notebook/ahskt_bd2006_notebook_metrics.json
manifest_path = /root/autodl-tmp/ahs-kt/outputs/ahskt_bd2006_notebook/ahskt_bd2006_notebook_manifest.json

Test metrics:


,loss,auc,acc,rmse
0,0.365246,0.808196,0.851128,0.333807



Split summary:


,item,value
0,train_visible_interactions,3673482
1,valid_targets_total,5717
2,test_targets_total,7672
3,train_sequences,18926
4,valid_sequences,5717
5,test_sequences,7671
6,valid_target_points_used,5717
7,valid_target_points_dropped_no_history,0
8,test_target_points_used,7671
9,test_target_points_dropped_no_history,1



Training history:


,epoch,train_auc,train_acc,train_rmse,valid_auc,valid_acc,valid_rmse
0,1,0.852864,0.902963,0.273866,0.795735,0.789050,0.381831
1,2,0.905716,0.913391,0.252343,0.816030,0.803218,0.370441
2,3,0.921672,0.919196,0.243316,0.817871,0.804968,0.366718


In [4]:
print('Notes:')
for note in result['prepared_metadata']['notes']:
    print('-', note)

print()
print('Saved bundle paths:')
for key, value in result['saved_paths'].items():
    print(f'- {key}: {value}')


Notes:
- KC(SubSkills) 为空时保留为 __NO_KC__，避免丢掉大量样本。
- validation 只取每个学生 train 时间轴最后若干个点，并且不把这些 held-out 点当作可观察历史。
- test 使用官方 master 标签，但每个测试点只允许看到更早的官方 train 交互。

Saved bundle paths:
- train_npz: data/ahskt_bd2006_notebook/ahskt_bd2006_notebook_train_ahskt.npz
- valid_npz: data/ahskt_bd2006_notebook/ahskt_bd2006_notebook_valid_ahskt.npz
- test_npz: data/ahskt_bd2006_notebook/ahskt_bd2006_notebook_test_ahskt.npz
- metadata_json: data/ahskt_bd2006_notebook/ahskt_bd2006_notebook_metadata.json
